# V3: temporal-trajectory ablation

Compares static_lr with temporal_lr, keeping the model type, outer folds, fixed parameters and imputation rules the same.

Variable missing rates are computed first in each training fold, then imputation, scaling and LASSO are trained. The established exemption is kept only for the V3 static white-cell differential; temporal slopes with too much missingness are still dropped. No patient is removed to obtain complete trajectories.

The difference measures the size of change; the slope the rate of change per hour; SD the fluctuation; the late-minus-early half-day difference the change between early and late phases; count the measurement intensity. Static mean/min/max may be correlated with the trajectories; LASSO λ.1se is the comparable main analysis, and temporal features are not guaranteed to improve performance.

**balanced simplified-compute version**: fixed base-model parameters, full keeps the 5×10 outer validation; LASSO inner 5 folds/30 λ, stacking inner 3 folds. Uses a cache separate from the parent strict version; the old predictions of the first 6 folds are not mixed in. No real patient model of the new version has been run yet.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Locate the project from the extension directory or the notebooks subdirectory, to avoid hard-coding a Windows drive letter.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dn_extensions" / "settings.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("start from dn_mortality_repro_V3_extensions or its notebooks subdirectory.")
sys.path.insert(0, str(ROOT))
print("extension directory:", ROOT)


In [ ]:
# pilot is a small 1×3 outer-fold trial run; for the formal experiment set PROFILE to "full".
# Starts only when RUN_EXPERIMENT is set to True; full may take a long time.
PROFILE = "pilot"
RUN_EXPERIMENT = False
EXPERIMENTS = ['static_lr', 'temporal_lr']
if RUN_EXPERIMENT:
    from dn_extensions.evaluation import run_evaluation
    from dn_extensions.settings import profile_options
    output, summary, comparisons = run_evaluation(profile_options(PROFILE), EXPERIMENTS)
    display(summary)
    display(comparisons)
else:
    print("model training in this notebook has not been started.")


In [ ]:
# Show the formal full results first; without results print the status, no retraining.
RESULT_PROFILE = "full"
latest = ROOT / "results" / RESULT_PROFILE / "latest.txt"
if latest.exists():
    folder = latest.parent / latest.read_text().strip()
    summary = pd.read_csv(folder / "summary.csv")
    display(summary[summary.experiment.isin(EXPERIMENTS)])
    print("detailed record:", folder / "report.md")
else:
    print("no experiment results for this mode yet. Set RESULT_PROFILE to pilot to show pilot results.")


## Experiment status and conclusions (updated automatically)

Run mode: `full`; method: balanced fixed parameters; result directory: `results/full/27464ba98305edbc`.

LASSO 5 folds/30 λ, stacking 3 folds; static and extension models use the same simplified method. No folds of the strict version were mixed in.

The fixed held-out set took no part in this experiment. Tree-model extensions need no internal stacking cross-fitting; the original fixed parameters and within-training-fold preprocessing are unchanged.

The Holm correction covers the paired comparisons currently available; corrected p-values may change when experiments are added later.

```text

  experiment  n_outer_folds  AUROC_fold_mean  AUROC_fold_sd  AUPRC_fold_mean  Brier_fold_mean  calibration_slope_fold_mean  seconds_total  AUROC_pooled_mean  AUPRC_pooled_mean  Brier_pooled_mean  cal_intercept_pooled_mean  cal_slope_pooled_mean
   static_lr             50          0.79928        0.02648          0.60740          0.15092                      0.92073      502.87958            0.79885            0.59482            0.15091                    0.00385                0.89831
   static_rf             50          0.79798        0.02809          0.60618          0.15177                      1.19841       15.22184            0.79671            0.59435            0.15177                    0.00179                1.16211
  static_xgb             50          0.79504        0.02965          0.60572          0.15250                      0.94083        4.61352            0.79408            0.59662            0.15249                    0.00766                0.91690
 temporal_lr             50          0.79564        0.02573          0.60179          0.15263                      0.88260     5687.76804            0.79475            0.58940            0.15262                    0.00314                0.85879
 temporal_rf             50          0.79741        0.02891          0.60594          0.15212                      1.24090       28.89751            0.79605            0.59470            0.15211                    0.00148                1.20217
temporal_xgb             50          0.79284        0.02796          0.60391          0.15300                      0.94159        8.29193            0.79177            0.59437            0.15299                    0.01106                0.91792

```

Paired differences:

```text

  experiment  reference metric  mean_diff  sd_diff  se_corrected       lo      hi        t  df  p_value  k  r  p_holm_current_family
   static_rf  static_lr  AUROC   -0.00130  0.01457       0.00528 -0.01190 0.00931 -0.24552  49  0.80708 10  5                    1.0
   static_rf  static_lr  AUPRC   -0.00122  0.02900       0.01050 -0.02232 0.01989 -0.11569  49  0.90837 10  5                    1.0
   static_rf  static_lr  Brier    0.00085  0.00469       0.00170 -0.00256 0.00427  0.50207  49  0.61787 10  5                    1.0
  static_xgb  static_lr  AUROC   -0.00423  0.01409       0.00510 -0.01449 0.00602 -0.82968  49  0.41074 10  5                    1.0
  static_xgb  static_lr  AUPRC   -0.00167  0.02894       0.01048 -0.02273 0.01938 -0.15972  49  0.87376 10  5                    1.0
  static_xgb  static_lr  Brier    0.00158  0.00519       0.00188 -0.00220 0.00536  0.83926  49  0.40540 10  5                    1.0
 temporal_lr  static_lr  AUROC   -0.00364  0.00867       0.00314 -0.00995 0.00267 -1.15931  49  0.25195 10  5                    1.0
 temporal_lr  static_lr  AUPRC   -0.00560  0.01502       0.00544 -0.01653 0.00532 -1.03037  49  0.30790 10  5                    1.0
 temporal_lr  static_lr  Brier    0.00171  0.00303       0.00110 -0.00049 0.00391  1.56078  49  0.12501 10  5                    1.0
 temporal_rf  static_rf  AUROC   -0.00057  0.00733       0.00265 -0.00591 0.00476 -0.21556  49  0.83022 10  5                    1.0
 temporal_rf  static_rf  AUPRC   -0.00024  0.01373       0.00497 -0.01023 0.00975 -0.04896  49  0.96115 10  5                    1.0
 temporal_rf  static_rf  Brier    0.00034  0.00196       0.00071 -0.00108 0.00177  0.48410  49  0.63047 10  5                    1.0
temporal_xgb static_xgb  AUROC   -0.00220  0.01022       0.00370 -0.00964 0.00523 -0.59579  49  0.55405 10  5                    1.0
temporal_xgb static_xgb  AUPRC   -0.00181  0.02169       0.00785 -0.01759 0.01397 -0.23090  49  0.81836 10  5                    1.0
temporal_xgb static_xgb  Brier    0.00050  0.00389       0.00141 -0.00233 0.00334  0.35547  49  0.72376 10  5                    1.0

```

- `static_rf` vs `static_lr`: ΔAUROC=-0.0013, corrected 95% CI [-0.0119, 0.0093]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `static_xgb` vs `static_lr`: ΔAUROC=-0.0042, corrected 95% CI [-0.0145, 0.0060]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `temporal_lr` vs `static_lr`: ΔAUROC=-0.0036, corrected 95% CI [-0.0099, 0.0027]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `temporal_rf` vs `static_rf`: ΔAUROC=-0.0006, corrected 95% CI [-0.0059, 0.0048]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

- `temporal_xgb` vs `static_xgb`: ΔAUROC=-0.0022, corrected 95% CI [-0.0096, 0.0052]; no evidence of a difference confirmed after this round of multiple-comparison correction. The Holm-corrected p-value, AUPRC, calibration and DCA should also be checked.

## 2026-10-14 review of the temporal ablation

The static and temporal models each completed 5×10 outer folds, using the same development set, fixed model parameters and outer split, without reading the held-out labels.

| Model | AUROC fold mean | AUPRC pooled OOF mean | Brier pooled OOF mean | Calibration slope pooled OOF mean |
|---|---:|---:|---:|---:|
| static_lr | 0.7993 | 0.5948 | 0.1509 | 0.898 |
| temporal_lr | 0.7956 | 0.5894 | 0.1526 | 0.859 |

`temporal_lr` vs `static_lr`: ΔAUROC=-0.0036, Nadeau–Bengio corrected 95% CI [-0.0099, 0.0027], p=0.252, Holm-corrected p=1.000.

**The 0–24 h trajectories as currently defined show no incremental predictive value in this configuration.** The pooled OOF AUROC is lower than static LR in all five repeats; 17 of the 50 folds are higher and 33 lower. But the corrected interval crosses 0, so a significant drop in performance cannot be claimed. Lower AUPRC, higher Brier and a calibration slope further from 1 also give no overall evidence in favour of keeping this feature set.

Implementation check: the median number of temporal features selected per fold is 4.5, so they are not all dropped. For example spo2_std is selected in 48/50 folds and temperature_count in 42/50. The number of measurements reflects testing intensity and cannot be interpreted directly as a physiological change or a modifiable factor.

This experiment tests only the current window, summarised trajectories, the 20% missingness rule, LASSO and fixed-parameter LR; it does not mean that all temporal modelling approaches are useless. Possible redundancy, missingness or increased model variance are explanatory hypotheses not yet verified mechanistically. The window or thresholds are not changed now in pursuit of a higher AUC.
